# Lab 8 - DuckDB: viajes de taxi de Nueva York


Repositorio del equipo: https://github.com/erickguerra2/duckdb, fork de https://github.com/menene/duckdb


In [1]:
import os
import sys
import json
import time
import shutil
import platform
import subprocess
import textwrap
import importlib.metadata as md
from pathlib import Path

import duckdb
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Image

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)

DIR_SQL = Path("sql")
DIR_FIG = Path("docs/figuras")
DIR_FIG.mkdir(parents=True, exist_ok=True)
DB_TAXI = Path("data/processed/taxi.duckdb")
METABASE = os.environ.get("MB_URL", "http://metabase:3000")

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Raíz del proyecto:", RAIZ)

Raíz del proyecto: /workspace


In [2]:
# Paleta validada
COLOR_TAXI = {"yellow": "#eda100", "green": "#008300"}
COLOR_ANIO = {"2024": "#2a78d6", "2025": "#eb6834", "2026": "#1baf7a"}
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
TINTA, TINTA_2, REJILLA = "#0b0b0b", "#52514e", "#e4e3df"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 130, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "regular",
    "axes.edgecolor": REJILLA, "axes.labelcolor": TINTA_2, "axes.titlecolor": TINTA,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": REJILLA, "grid.linewidth": 0.6,
    "xtick.color": TINTA_2, "ytick.color": TINTA_2,
    "lines.linewidth": 2, "legend.frameon": False,
})
miles = FuncFormatter(lambda x, _: f"{x:,.0f}")


def guardar(fig, nombre):
    fig.savefig(DIR_FIG / f"{nombre}.png")
    plt.show()

In [3]:
CATALOGO = []
FUENTE = {"texto": ""}
con = duckdb.connect()


def consulta(nombre, objetivo, sql, fuente=None, conexion=None, mostrar=True):
    conexion = conexion or con
    fuente = fuente or FUENTE["texto"]
    sql = textwrap.dedent(sql).strip().rstrip(";")
    ruta = DIR_SQL / f"{nombre}.sql"
    ruta.parent.mkdir(parents=True, exist_ok=True)
    ruta.write_text(f"-- Objetivo: {objetivo}\n-- Fuente: {fuente}\n{sql};\n", encoding="utf-8")
    inicio = time.perf_counter()
    df = conexion.sql(sql).df()
    segundos = time.perf_counter() - inicio
    CATALOGO.append({"consulta": nombre, "objetivo": objetivo, "fuente": fuente,
                     "filas": len(df), "segundos": round(segundos, 2)})
    print(f"[{ruta}]  {len(df)} filas en {segundos:.2f} s")
    print(f"Objetivo: {objetivo}")
    print(f"Fuente: {fuente}")
    if mostrar:
        display(df)
    return df


def descargar(*anios):
    args = [sys.executable, "scripts/download_data.py"] + (["--anio", *map(str, anios)] if anios else [])
    r = subprocess.run(args, capture_output=True, text=True)
    lineas = r.stdout.splitlines()
    omitidos = sum("ya existe" in l for l in lineas)
    nuevos = sum("listo (" in l for l in lineas)
    print(f"Archivos omitidos por existir: {omitidos} | descargados ahora: {nuevos}")
    print("\n".join(lineas[lineas.index("RESUMEN") - 1:]))
    return pd.read_csv("docs/manifest_descargas.csv")

## Ejercicio 1. Preparación del ambiente

### 1.1 y 1.2 Fork, clonación y arranque

El repositorio del docente se bifurcó en https://github.com/erickguerra2/duckdb 

### Estructura del proyecto

| Directorio o archivo | Propósito |
|---|---|
| data/raw | Archivos Parquet tal como los publica la TLC, organizados en tipo y año. Nunca se modifican |
| data/processed | Productos derivados: la base taxi.duckdb materializada y archivos temporales del benchmark |
| notebooks | Este notebook con el análisis, que documenta cada consulta y su interpretación |
| scripts | Código reutilizable: descarga, vistas y materialización, benchmark y publicación del tablero |
| sql | Todas las consultas SQL versionadas: vistas, consultas de cada ejercicio, benchmark e indicadores |
| docs | Evidencia generada: manifiesto de descarga, bitácoras, resultados del benchmark, figuras y capturas del tablero |
| Dockerfile | Imagen del ambiente de análisis con Python y las librerías fijadas en requirements.txt |
| metabase.Dockerfile | Imagen de Metabase con el driver de DuckDB en versiones alineadas |
| docker-compose.yml | Orquesta ambos servicios, puertos y volúmenes |
| README.md | Instrucciones para reproducir todo el trabajo |

La separación entre raw y processed permite regenerar cualquier producto desde los datos originales, y data está excluida de Git por .gitignore.

### 1.3 y 1.4 Verificación de servicios y herramientas disponibles

In [4]:
print("Python", platform.python_version(), "|", platform.platform())
for paquete in ["duckdb", "pandas", "pyarrow", "matplotlib", "requests", "jupyterlab", "nbconvert"]:
    print(f"  {paquete:12} {md.version(paquete)}")
print("DuckDB SQL:", duckdb.sql("SELECT version()").fetchone()[0])
print("curl:", shutil.which("curl"))

salud = requests.get(f"{METABASE}/api/health", timeout=10).json()
print("Metabase /api/health:", salud)
version_mb = requests.get(f"{METABASE}/api/session/properties", timeout=10).json()["version"]["tag"]
print("Metabase versión:", version_mb)
print("Driver DuckDB en Metabase:", "duckdb" in requests.get(f"{METABASE}/api/session/properties").json()["engines"])

for carpeta in ["data/raw", "data/processed", "notebooks", "scripts", "sql", "docs"]:
    print(f"  montaje {carpeta:15} existe={Path(carpeta).exists()}")

Python 3.11.14 | Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.41
  duckdb       1.5.5
  pandas       3.0.6
  pyarrow      25.0.1
  matplotlib   3.11.2
  requests     2.34.2
  jupyterlab   4.6.4
  nbconvert    7.17.1
DuckDB SQL: v1.5.5
curl: /usr/bin/curl
Metabase /api/health: {'status': 'ok'}


Metabase versión: v0.63.19


Driver DuckDB en Metabase: True
  montaje data/raw        existe=True
  montaje data/processed  existe=True
  montaje notebooks       existe=True
  montaje scripts         existe=True
  montaje sql             existe=True
  montaje docs            existe=True


Herramientas disponibles dentro del ambiente:

| Herramienta | Uso en el laboratorio |
|---|---|
| Python 3.11 | Lenguaje de los scripts y del notebook |
| DuckDB 1.5.5 | Motor analítico en proceso: consulta Parquet y materializa tablas |
| pandas y pyarrow | Reciben resultados de DuckDB y leen metadatos Parquet sin cargar los datos |
| matplotlib | Gráficos del notebook |
| requests y curl | Descarga desde la TLC y uso de la API de Metabase |
| JupyterLab y nbconvert | Edición y ejecución reproducible del notebook |
| Metabase con driver DuckDB | Herramienta de tableros proporcionada, conectada a taxi.duckdb en modo solo lectura |

Ambos servicios responden: JupyterLab sirve este notebook y Metabase devuelve status ok con el driver de DuckDB registrado.

### 1.5 Procedimiento documentado

El procedimiento para levantar el ambiente quedó en la sección Como levantar el ambiente del README.md.

### 1.6 Por qué un ambiente reproducible

Un análisis de datos depende de versiones exactas: un cambio de DuckDB puede alterar el formato del archivo .duckdb, la forma de inferir tipos o el resultado de funciones como quantile_cont, y Metabase solo abre bases creadas con la misma versión de DuckDB que su driver. Con Docker cualquier integrante o el docente obtiene las mismas versiones de Python, librerías y servicios con un solo comando, sin depender de lo instalado en cada computadora. Eso hace que los resultados se puedan verificar, que los errores se puedan reproducir y que el trabajo siga funcionando meses después. También aísla dependencias pesadas como Java para Metabase y deja documentada la infraestructura como código.

## Ejercicio 2. Sistema de descarga

### 2.1 Análisis del script original

El script scripts/download_data.py entregado ya resolvía bien varias cosas: consultaba con HEAD si un archivo estaba publicado, descargaba por bloques sobre un archivo temporal .part con reintentos y omitía los archivos existentes. Sus limitaciones eran:

- El año estaba fijo en la constante ANIO = 2026 y se usaba dentro de construir_nombre, construir_url y ruta_destino, así que agregar 2024 o 2025 obligaba a editar el código.
- No había un parámetro de línea de comandos para elegir años.
- Recorría siempre los 12 meses, incluso meses futuros que no pueden existir.
- No verificaba que el tamaño descargado coincidiera con el publicado, por lo que un corte de red podía dejar un archivo truncado que después se consideraría existente.
- No había forma de comprobar al final que el conjunto descargado estuviera completo ni registro de lo obtenido.
- No descargaba el catálogo de zonas, necesario para traducir PULocationID y DOLocationID a nombres.

### 2.2 a 2.4 Cambios realizados

- Las funciones reciben el año como parámetro y se agregó --anio, que acepta uno o varios años. Por defecto descarga 2026.
- meses_posibles limita el año en curso al mes actual.
- La descarga compara los bytes escritos con Content-Length y descarta el archivo si no coinciden.
- Se agregó --verificar para validar sin descargar.
- Al terminar, validar_local revisa cada archivo: tamaño local contra tamaño publicado, lectura de los metadatos Parquet con pyarrow, número de filas y columnas, y huecos de meses intermedios. El resultado se guarda en docs/manifest_descargas.csv.
- Se descarga una sola vez data/raw/zones/taxi_zone_lookup.csv.
- Se conservó la estructura data/raw/tipo/año/nombre-original.parquet y la regla de no volver a descargar archivos existentes.

### 2.5 Ejecución

La primera ejecución real quedó registrada en docs/logs/descarga_2026.txt. Aquí se muestra ese registro y luego se ejecuta otra vez el script para comprobar que no vuelve a descargar nada.

In [5]:
primera = Path("docs/logs/descarga_2026.txt").read_text().splitlines()
print("\n".join(primera[:12] + ["  ..."] + primera[-16:]))


=== YELLOW 2026 ===
  2026-01  descargando...
  2026-01  listo (61.2 MiB) -> data/raw/yellow/2026/yellow_tripdata_2026-01.parquet
  2026-02  descargando...
  2026-02  listo (56.0 MiB) -> data/raw/yellow/2026/yellow_tripdata_2026-02.parquet
  2026-03  descargando...
  2026-03  listo (64.7 MiB) -> data/raw/yellow/2026/yellow_tripdata_2026-03.parquet
  2026-04  descargando...
  2026-04  listo (61.8 MiB) -> data/raw/yellow/2026/yellow_tripdata_2026-04.parquet
  2026-05  descargando...
  2026-05  listo (66.5 MiB) -> data/raw/yellow/2026/yellow_tripdata_2026-05.parquet
  ...
Validando archivos locales...

RESUMEN
  anios         : 2026
  descargados   : 16
  ya existian   : 0
  no publicados : 4
      yellow 2026-09, yellow 2026-10, green 2026-09, green 2026-10
  fallidos      : 0
  archivos locales validados : 16
  registros totales          : 30,040,469
  archivos con problemas     : 0
  manifiesto                 : docs/manifest_descargas.csv


In [6]:
manifiesto = descargar(2026)

Archivos omitidos por existir: 17 | descargados ahora: 0
RESUMEN
  anios         : 2026
  descargados   : 0
  ya existian   : 16
  no publicados : 4
      yellow 2026-09, yellow 2026-10, green 2026-09, green 2026-10
  fallidos      : 0
  archivos locales validados : 16
  registros totales          : 30,040,469
  archivos con problemas     : 0
  manifiesto                 : docs/manifest_descargas.csv


In [7]:
resumen_2026 = (manifiesto.groupby(["tipo", "anio"])
                .agg(archivos=("mes", "count"), primer_mes=("mes", "min"), ultimo_mes=("mes", "max"),
                     registros=("registros", "sum"), mib=("bytes_local", lambda b: b.sum() / 2**20),
                     tamanio_ok=("tamanio_ok", "all"), legibles=("parquet_legible", "all"))
                .reset_index())
display(resumen_2026)

,tipo,anio,archivos,primer_mes,ultimo_mes,registros,mib,tamanio_ok,legibles
0,green,2026,8,1,8,337114,7.89,True,True
1,yellow,2026,8,1,8,29703355,487.76,True,True


### 2.6 Documentación de los cambios

Los cambios están descritos en 2.2 a 2.4, en el docstring del script y en la sección Como descargar los datos del README.

### 2.7 Cómo se determinó que el conjunto está completo

Se usaron cuatro criterios independientes:

1. Publicación: para cada mes posible el script consulta al servidor de la TLC. En 2026 están publicados enero a agosto para ambos tipos; septiembre y octubre responden como no publicados porque la TLC publica con unas semanas de atraso. Ningún mes publicado quedó sin descargar.
2. Continuidad: no hay huecos de meses intermedios; los 8 meses son consecutivos para cada tipo.
3. Integridad de bytes: el tamaño local coincide exactamente con el Content-Length del servidor en los 16 archivos.
4. Integridad del formato: los 16 archivos abren como Parquet válido y reportan filas y columnas desde sus metadatos, sin errores.

Con eso hay 16 archivos y 30,040,469 registros de 2026. En el ejercicio 3 se vuelve a contar con DuckDB y el total coincide con los metadatos.

## Ejercicio 3. Consultas directas sobre archivos Parquet

En este ejercicio no se crea ninguna tabla ni vista: cada consulta usa read_parquet, glob, parquet_schema o parquet_metadata directamente sobre los archivos de 2026. Cada consulta se guarda automáticamente en sql/ejercicio3 con su objetivo y su fuente, y su resultado se muestra debajo.

In [8]:
YELLOW_26 = "data/raw/yellow/2026/*.parquet"
GREEN_26 = "data/raw/green/2026/*.parquet"
TODOS_26 = "data/raw/*/2026/*.parquet"
FUENTE["texto"] = "data/raw/{yellow,green}/2026/*.parquet"

### 3.1 Cantidad de archivos

In [9]:
archivos = consulta("ejercicio3/01_cantidad_archivos", "Contar los archivos Parquet de 2026 por tipo de taxi", f"""
    SELECT split_part(file, '/', 3) AS taxi,
           count(*) AS archivos,
           min(regexp_extract(file, '(\\d{{4}}-\\d{{2}})', 1)) AS primer_mes,
           max(regexp_extract(file, '(\\d{{4}}-\\d{{2}})', 1)) AS ultimo_mes
    FROM glob('{TODOS_26}')
    GROUP BY taxi
    ORDER BY taxi
""")

[sql/ejercicio3/01_cantidad_archivos.sql]  2 filas en 0.05 s
Objetivo: Contar los archivos Parquet de 2026 por tipo de taxi
Fuente: data/raw/{yellow,green}/2026/*.parquet


,taxi,archivos,primer_mes,ultimo_mes
0,green,8,2026-01,2026-08
1,yellow,8,2026-01,2026-08


### 3.2 Cantidad de registros

In [10]:
registros = consulta("ejercicio3/02_cantidad_registros", "Contar registros leyendo los archivos completos", f"""
    SELECT 'yellow' AS taxi, count(*) AS registros FROM read_parquet('{YELLOW_26}', union_by_name = true)
    UNION ALL
    SELECT 'green', count(*) FROM read_parquet('{GREEN_26}', union_by_name = true)
""")
metadatos = consulta("ejercicio3/03_registros_metadatos", "Contar registros solo desde los metadatos Parquet", f"""
    SELECT split_part(file_name, '/', 3) AS taxi,
           count(DISTINCT file_name) AS archivos,
           sum(row_group_num_rows) FILTER (WHERE column_id = 0) AS registros,
           count(*) FILTER (WHERE column_id = 0) AS row_groups
    FROM parquet_metadata('{TODOS_26}')
    GROUP BY taxi
    ORDER BY taxi
""")
print("Total 2026:", f"{registros.registros.sum():,}", "| coincide con manifiesto:",
      registros.registros.sum() == manifiesto.registros.sum())

[sql/ejercicio3/02_cantidad_registros.sql]  2 filas en 0.09 s
Objetivo: Contar registros leyendo los archivos completos
Fuente: data/raw/{yellow,green}/2026/*.parquet


,taxi,registros
0,yellow,29703355
1,green,337114


[sql/ejercicio3/03_registros_metadatos.sql]  2 filas en 0.05 s
Objetivo: Contar registros solo desde los metadatos Parquet
Fuente: data/raw/{yellow,green}/2026/*.parquet


,taxi,archivos,registros,row_groups
0,green,8,"337,114.00",8
1,yellow,8,"29,703,355.00",32


Total 2026: 30,040,469 | coincide con manifiesto: True


### 3.3 y 3.4 Columnas y tipos de datos

In [11]:
esquema = consulta("ejercicio3/04_columnas_por_archivo", "Columnas y tipo físico presentes en cada archivo", f"""
    SELECT split_part(file_name, '/', 3) AS taxi,
           name AS columna,
           type AS tipo_fisico,
           logical_type,
           count(DISTINCT file_name) AS archivos_con_columna,
           min(regexp_extract(file_name, '(\\d{{4}}-\\d{{2}})', 1)) AS desde
    FROM parquet_schema('{TODOS_26}')
    WHERE name <> 'schema'
    GROUP BY ALL
    ORDER BY taxi, columna
""", mostrar=False)
display(esquema.pivot_table(index="columna", columns="taxi", values="archivos_con_columna", fill_value=0))

[sql/ejercicio3/04_columnas_por_archivo.sql]  43 filas en 0.05 s
Objetivo: Columnas y tipo físico presentes en cada archivo
Fuente: data/raw/{yellow,green}/2026/*.parquet


taxi,green,yellow
columna,,
Airport_fee,0.00,8.00
DOLocationID,8.00,8.00
PULocationID,8.00,8.00
RatecodeID,8.00,8.00
VendorID,8.00,8.00
cbd_congestion_fee,8.00,8.00
congestion_surcharge,8.00,8.00
ehail_fee,8.00,0.00
extra,8.00,8.00


In [12]:
tipos_y = consulta("ejercicio3/05_tipos_yellow", "Tipos lógicos que DuckDB asigna a las columnas de taxis amarillos",
                   f"DESCRIBE SELECT * FROM read_parquet('{YELLOW_26}', union_by_name = true)", mostrar=False)
tipos_g = consulta("ejercicio3/06_tipos_green", "Tipos lógicos que DuckDB asigna a las columnas de taxis verdes",
                   f"DESCRIBE SELECT * FROM read_parquet('{GREEN_26}', union_by_name = true)", mostrar=False)
tipos = (tipos_y[["column_name", "column_type"]].rename(columns={"column_type": "yellow"})
         .merge(tipos_g[["column_name", "column_type"]].rename(columns={"column_type": "green"}),
                on="column_name", how="outer"))
display(tipos)

[sql/ejercicio3/05_tipos_yellow.sql]  21 filas en 0.04 s
Objetivo: Tipos lógicos que DuckDB asigna a las columnas de taxis amarillos
Fuente: data/raw/{yellow,green}/2026/*.parquet


[sql/ejercicio3/06_tipos_green.sql]  22 filas en 0.04 s
Objetivo: Tipos lógicos que DuckDB asigna a las columnas de taxis verdes
Fuente: data/raw/{yellow,green}/2026/*.parquet


,column_name,yellow,green
0,Airport_fee,DOUBLE,NaN
1,DOLocationID,INTEGER,INTEGER
2,PULocationID,INTEGER,INTEGER
3,RatecodeID,BIGINT,BIGINT
4,VendorID,INTEGER,INTEGER
5,cbd_congestion_fee,DOUBLE,DOUBLE
6,congestion_surcharge,DOUBLE,DOUBLE
7,ehail_fee,NaN,DOUBLE
8,extra,DOUBLE,DOUBLE
9,fare_amount,DOUBLE,DOUBLE


Los amarillos tienen 21 columnas y los verdes 22. Las diferencias de esquema son:

- Las fechas se llaman tpep_pickup_datetime y tpep_dropoff_datetime en amarillos, y lpep en verdes.
- Solo los amarillos tienen Airport_fee; solo los verdes tienen ehail_fee y trip_type.
- request_source aparece solo desde junio de 2026, en 3 de los 8 archivos de cada tipo. Es una columna nueva que no está en el diccionario de datos publicado por la TLC.
- passenger_count, RatecodeID y payment_type vienen como BIGINT aunque son códigos pequeños; las zonas son INTEGER y los montos DOUBLE.

Decisión: para analizar ambos tipos juntos se necesita una vista que renombre las fechas, homologue tipos y tolere columnas ausentes. Se construye en el ejercicio 4.

### 3.5 Muestra de registros

In [13]:
muestra = consulta("ejercicio3/07_muestra_yellow", "Muestra reproducible de viajes amarillos", f"""
    SELECT * FROM read_parquet('{YELLOW_26}', union_by_name = true)
    USING SAMPLE reservoir(5 ROWS) REPEATABLE (42)
""")
muestra_g = consulta("ejercicio3/08_muestra_green", "Muestra reproducible de viajes verdes", f"""
    SELECT * FROM read_parquet('{GREEN_26}', union_by_name = true)
    USING SAMPLE reservoir(5 ROWS) REPEATABLE (42)
""")

[sql/ejercicio3/07_muestra_yellow.sql]  5 filas en 0.21 s
Objetivo: Muestra reproducible de viajes amarillos
Fuente: data/raw/{yellow,green}/2026/*.parquet


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source
0,1,2026-01-01 00:34:45,2026-01-01 00:40:15,4,0.90,1,N,262,141,1,7.90,3.50,0.50,3.20,0.00,1.00,16.10,2.50,0.00,0.00,None
1,1,2026-01-01 00:06:07,2026-01-01 00:20:20,2,1.90,1,N,231,90,1,14.20,4.25,0.50,3.95,0.00,1.00,23.90,2.50,0.00,0.75,None
2,2,2026-01-01 03:15:42,2026-01-01 03:20:33,1,1.15,1,N,263,140,1,7.20,1.00,0.50,2.44,0.00,1.00,14.64,2.50,0.00,0.00,None
3,2,2026-01-01 04:25:35,2026-01-01 04:38:18,1,3.69,1,N,107,143,1,17.70,1.00,0.50,4.69,0.00,1.00,28.14,2.50,0.00,0.75,None
4,2,2026-01-01 06:33:56,2026-01-01 06:55:37,2,10.27,2,N,68,138,2,70.00,0.00,0.50,0.00,6.94,1.00,81.69,2.50,0.00,0.75,None


[sql/ejercicio3/08_muestra_green.sql]  5 filas en 0.11 s
Objetivo: Muestra reproducible de viajes verdes
Fuente: data/raw/{yellow,green}/2026/*.parquet


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,request_source
0,2,2026-01-02 05:43:34,2026-01-02 05:44:09,N,5,34,34,2,0.40,70.00,0.00,0.00,14.20,0.00,NaN,1.00,85.20,1,2,0.00,0.00,None
1,2,2026-01-02 21:02:00,2026-01-02 21:07:42,N,1,74,41,1,1.17,7.90,1.00,0.50,2.08,0.00,NaN,1.00,12.48,1,1,0.00,0.00,None
2,1,2026-01-14 21:03:03,2026-01-14 21:06:16,N,1,181,65,1,0.60,5.80,1.00,1.50,0.00,0.00,NaN,1.00,8.30,2,1,0.00,0.00,None
3,2,2026-01-16 19:11:20,2026-01-16 19:29:44,N,1,74,151,1,2.33,17.70,2.50,0.50,4.34,0.00,NaN,1.00,26.04,1,1,0.00,0.00,None
4,2,2026-01-19 02:15:45,2026-01-19 02:20:57,N,1,95,102,1,1.35,8.60,1.00,0.50,1.00,0.00,NaN,1.00,12.10,1,1,0.00,0.00,None


### 3.6 Problemas de calidad

Primero se usa SUMMARIZE, que calcula en una sola pasada mínimo, máximo, valores distintos aproximados, promedio, cuartiles y porcentaje de nulos de cada columna.

In [14]:
perfil_y = consulta("ejercicio3/09_perfil_yellow", "Perfil estadístico de todas las columnas amarillas",
                    f"SUMMARIZE SELECT * FROM read_parquet('{YELLOW_26}', union_by_name = true)", mostrar=False)
display(perfil_y[["column_name", "min", "max", "approx_unique", "avg", "q50", "null_percentage"]])

[sql/ejercicio3/09_perfil_yellow.sql]  21 filas en 11.44 s
Objetivo: Perfil estadístico de todas las columnas amarillas
Fuente: data/raw/{yellow,green}/2026/*.parquet


,column_name,min,max,approx_unique,avg,q50,null_percentage
0,VendorID,1,7,4,1.885739809526567,2,0.00
1,tpep_pickup_datetime,2001-01-01 09:23:58,2026-08-31 23:59:59,16867928,2026-04-30 15:29:29.519432,2026-04-30 23:13:00.550942,0.00
2,tpep_dropoff_datetime,2001-01-01 16:09:38,2026-09-01 20:16:00,16587125,2026-04-30 15:47:08.186216,2026-04-30 19:30:25.715078,0.00
3,passenger_count,0,9,11,1.2494318488564,1,25.98
4,trip_distance,0.0,328522.2,7217,5.552940149689086,1.8562155905933497,0.00
5,RatecodeID,1,99,7,4.527471444398553,1,25.98
6,store_and_fwd_flag,N,Y,2,NaN,NaN,25.98
7,PULocationID,1,265,290,161.57793013617484,161,0.00
8,DOLocationID,1,265,298,161.05139342003622,162,0.00
9,payment_type,0,5,6,0.8621735154160195,1,0.00


In [15]:
perfil_g = consulta("ejercicio3/10_perfil_green", "Perfil estadístico de todas las columnas verdes",
                    f"SUMMARIZE SELECT * FROM read_parquet('{GREEN_26}', union_by_name = true)", mostrar=False)
display(perfil_g[["column_name", "min", "max", "approx_unique", "avg", "q50", "null_percentage"]])

[sql/ejercicio3/10_perfil_green.sql]  22 filas en 0.33 s
Objetivo: Perfil estadístico de todas las columnas verdes
Fuente: data/raw/{yellow,green}/2026/*.parquet


,column_name,min,max,approx_unique,avg,q50,null_percentage
0,VendorID,1,6,3,2.328351833504393,2,0.00
1,lpep_pickup_datetime,2008-12-31 17:35:31,2026-08-31 23:58:28,332702,2026-05-02 13:25:51.096427,2026-05-03 11:25:12.16329,0.00
2,lpep_dropoff_datetime,2008-12-31 23:16:26,2026-09-02 09:39:37,396286,2026-05-02 13:46:38.833759,2026-05-03 22:53:23.532589,0.00
3,store_and_fwd_flag,N,Y,2,NaN,NaN,14.47
4,RatecodeID,1,99,7,1.2549672434183374,1,14.47
5,PULocationID,1,265,263,97.32761024460568,75,0.00
6,DOLocationID,1,265,266,142.8768458147689,140,0.00
7,passenger_count,0,9,11,1.3006461144694266,1,14.47
8,trip_distance,0.0,179830.92,2472,13.349507644298379,2.0659214503119703,0.00
9,fare_amount,-500.0,1676.7,4808,17.014101995170655,13.164726506912354,0.00


In [16]:
fechas = consulta("ejercicio3/11_fechas_fuera_de_archivo", "Viajes cuya fecha no corresponde al mes del archivo", f"""
    SELECT split_part(filename, '/', 3) AS taxi,
           count(*) AS registros,
           count_if(strftime(coalesce(tpep_pickup_datetime, lpep_pickup_datetime), '%Y-%m')
                    <> regexp_extract(filename, '(\\d{{4}}-\\d{{2}})', 1)) AS fuera_de_mes,
           min(coalesce(tpep_pickup_datetime, lpep_pickup_datetime)) AS fecha_minima,
           max(coalesce(tpep_pickup_datetime, lpep_pickup_datetime)) AS fecha_maxima
    FROM read_parquet('{TODOS_26}', union_by_name = true, filename = true)
    GROUP BY taxi
""")

[sql/ejercicio3/11_fechas_fuera_de_archivo.sql]  2 filas en 0.88 s
Objetivo: Viajes cuya fecha no corresponde al mes del archivo
Fuente: data/raw/{yellow,green}/2026/*.parquet


,taxi,registros,fuera_de_mes,fecha_minima,fecha_maxima
0,green,337114,98.00,2008-12-31 17:35:31,2026-08-31 23:58:28
1,yellow,29703355,146.00,2001-01-01 09:23:58,2026-08-31 23:59:59


In [17]:
reglas = consulta("ejercicio3/12_reglas_inconsistencia", "Contar registros que violan reglas básicas de negocio", f"""
    WITH v AS (
        SELECT split_part(filename, '/', 3) AS taxi,
               coalesce(tpep_pickup_datetime, lpep_pickup_datetime) AS pickup,
               coalesce(tpep_dropoff_datetime, lpep_dropoff_datetime) AS dropoff,
               *
        FROM read_parquet('{TODOS_26}', union_by_name = true, filename = true)
    )
    SELECT taxi,
           count(*) AS registros,
           count_if(dropoff <= pickup) AS duracion_no_positiva,
           count_if(dropoff - pickup > INTERVAL 6 HOUR) AS duracion_mayor_6h,
           count_if(trip_distance = 0) AS distancia_cero,
           count_if(trip_distance > 100) AS distancia_mayor_100,
           count_if(total_amount < 0) AS total_negativo,
           count_if(total_amount = 0) AS total_cero,
           count_if(total_amount > 1000) AS total_mayor_1000,
           count_if(passenger_count = 0) AS pasajeros_cero,
           count_if(passenger_count IS NULL) AS pasajeros_nulo,
           count_if(PULocationID IN (264, 265)) AS zona_desconocida
    FROM v
    GROUP BY taxi
""")

[sql/ejercicio3/12_reglas_inconsistencia.sql]  2 filas en 1.58 s
Objetivo: Contar registros que violan reglas básicas de negocio
Fuente: data/raw/{yellow,green}/2026/*.parquet


,taxi,registros,duracion_no_positiva,duracion_mayor_6h,distancia_cero,distancia_mayor_100,total_negativo,total_cero,total_mayor_1000,pasajeros_cero,pasajeros_nulo,zona_desconocida
0,green,337114,234.00,"1,104.00","12,212.00",72.00,"1,023.00",543.00,1.00,"4,527.00","48,775.00","1,131.00"
1,yellow,29703355,"371,683.00","7,315.00","952,231.00","1,223.00","161,835.00","5,258.00",49.00,"91,359.00","7,716,688.00","49,676.00"


In [18]:
codigos = consulta("ejercicio3/13_codigos_categoricos", "Revisar códigos contra el diccionario de la TLC", f"""
    SELECT 'payment_type' AS campo, payment_type::VARCHAR AS codigo, count(*) AS registros
    FROM read_parquet('{YELLOW_26}', union_by_name = true) GROUP BY ALL
    UNION ALL
    SELECT 'RatecodeID', RatecodeID::VARCHAR, count(*)
    FROM read_parquet('{YELLOW_26}', union_by_name = true) GROUP BY ALL
    UNION ALL
    SELECT 'VendorID', VendorID::VARCHAR, count(*)
    FROM read_parquet('{YELLOW_26}', union_by_name = true) GROUP BY ALL
    ORDER BY campo, codigo
""")

[sql/ejercicio3/13_codigos_categoricos.sql]  18 filas en 0.59 s
Objetivo: Revisar códigos contra el diccionario de la TLC
Fuente: data/raw/{yellow,green}/2026/*.parquet


,campo,codigo,registros
0,RatecodeID,1,20102072
1,RatecodeID,2,694936
2,RatecodeID,3,90052
3,RatecodeID,4,67285
4,RatecodeID,5,262614
5,RatecodeID,6,15
6,RatecodeID,99,769693
7,RatecodeID,NaN,7716688
8,VendorID,1,5467071
9,VendorID,2,23809774


In [19]:
flex = consulta("ejercicio3/14_nulos_flex_fare", "Verificar si los nulos se concentran en un tipo de pago", f"""
    SELECT payment_type,
           count(*) AS registros,
           count_if(passenger_count IS NULL) AS pasajeros_nulo,
           count_if(RatecodeID IS NULL) AS ratecode_nulo,
           count_if(store_and_fwd_flag IS NULL) AS flag_nulo,
           count_if(Airport_fee IS NULL) AS airport_fee_nulo,
           round(avg(total_amount), 2) AS total_promedio
    FROM read_parquet('{YELLOW_26}', union_by_name = true)
    GROUP BY payment_type
    ORDER BY payment_type
""")

[sql/ejercicio3/14_nulos_flex_fare.sql]  6 filas en 0.49 s
Objetivo: Verificar si los nulos se concentran en un tipo de pago
Fuente: data/raw/{yellow,green}/2026/*.parquet


,payment_type,registros,pasajeros_nulo,ratecode_nulo,flag_nulo,airport_fee_nulo,total_promedio
0,0,7716688,"7,716,688.00","7,716,688.00","7,716,688.00","7,716,688.00",32.57
1,1,18941008,0.00,0.00,0.00,0.00,30.18
2,2,2708031,0.00,0.00,0.00,0.00,25.17
3,3,98138,0.00,0.00,0.00,0.00,12.67
4,4,239488,0.00,0.00,0.00,0.00,3.47
5,5,2,0.00,0.00,0.00,0.00,0.00


In [20]:
origen = consulta("ejercicio3/15_request_source", "Explorar la columna nueva request_source por mes", f"""
    SELECT split_part(filename, '/', 3) AS taxi,
           regexp_extract(filename, '(\\d{{4}}-\\d{{2}})', 1) AS mes,
           coalesce(request_source, 'NULL') AS request_source,
           count(*) AS registros
    FROM read_parquet('{TODOS_26}', union_by_name = true, filename = true)
    WHERE regexp_extract(filename, '(\\d{{4}}-\\d{{2}})', 1) >= '2026-05'
    GROUP BY ALL
    ORDER BY taxi, mes, registros DESC
""", mostrar=False)
display(origen.pivot_table(index=["taxi", "request_source"], columns="mes", values="registros", fill_value=0))

[sql/ejercicio3/15_request_source.sql]  29 filas en 0.32 s
Objetivo: Explorar la columna nueva request_source por mes
Fuente: data/raw/{yellow,green}/2026/*.parquet


mes                        2026-05      2026-06      2026-07      2026-08
taxi   request_source                                                    
green  A                      0.00     6,471.00     6,428.00     5,130.00
       CC                     0.00         0.00         2.00         1.00
       HV0005                 0.00         0.00         0.00     1,179.00
       NULL              44,921.00    37,692.00    34,822.00    34,377.00
yellow A                      0.00    76,711.00   223,309.00   103,753.00
       CC                     0.00       664.00       642.00       573.00
       EH0004                 0.00     8,107.00     6,409.00     6,253.00
       EH0010                 0.00         0.00         9.00       263.00
       HV0003                 0.00   927,698.00   739,048.00   628,774.00
       HV0005                 0.00         0.00         0.00   181,233.00
       NULL           4,090,836.00 2,824,068.00 2,560,692.00 2,415,867.00

In [21]:
duplicados = consulta("ejercicio3/16_duplicados", "Detectar viajes repetidos con la misma huella", f"""
    SELECT count(*) AS grupos_repetidos, coalesce(sum(n - 1), 0) AS filas_sobrantes
    FROM (
        SELECT VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, PULocationID, DOLocationID,
               trip_distance, total_amount, count(*) AS n
        FROM read_parquet('{YELLOW_26}', union_by_name = true)
        GROUP BY ALL
        HAVING count(*) > 1
    )
""")

[sql/ejercicio3/16_duplicados.sql]  1 filas en 1.88 s
Objetivo: Detectar viajes repetidos con la misma huella
Fuente: data/raw/{yellow,green}/2026/*.parquet


,grupos_repetidos,filas_sobrantes
0,32790,"32,790.00"


Problemas de calidad observados:

1. Fechas fuera del archivo: 146 viajes amarillos y 98 verdes tienen una fecha que no corresponde al mes del archivo. Hay fechas desde 2001 en amarillos y desde 2008 en verdes, que son errores del reloj del taxímetro.
2. Montos negativos: 161,835 viajes amarillos y 1,023 verdes tienen total negativo, con mínimos de -2,560 y -501 dólares. Parecen reversos o correcciones contables, no viajes.
3. Distancia cero o imposible: 952,231 viajes amarillos, el 3.2 %, y 12,212 verdes tienen distancia cero. El máximo es de 328,522 millas en 22 minutos.
4. Duración no positiva: 371,683 viajes amarillos terminan antes o en el mismo segundo en que empiezan. En el ejercicio 4 se ve que casi todos vienen de un solo proveedor.
5. Nulos estructurales: el 25.98 % de los amarillos tiene nulos en passenger_count, RatecodeID, store_and_fwd_flag, congestion_surcharge y Airport_fee. Los 7,716,688 nulos coinciden exactamente con payment_type 0, que el diccionario de la TLC define como Flex Fare, un viaje con precio fijo acordado antes de empezar. No son datos perdidos sino campos que no aplican. En verdes pasa lo mismo con el 14.47 % de los registros, que no traen payment_type.
6. Códigos especiales: 769,693 amarillos tienen RatecodeID 99, que significa desconocido. Los VendorID 6 y 7 son válidos según el diccionario de marzo de 2025.
7. Columna no documentada: request_source aparece desde junio de 2026 y no está en el diccionario. Sus valores HV0003 y HV0005 coinciden con las licencias de Uber y Lyft en los datos de vehículos de alto volumen de la TLC, y EH0004 y EH0010 parecen aplicaciones de e-hail; A y CC no tienen explicación publicada. En junio a agosto el 23 % de los viajes amarillos tiene un código HV, es decir, se pidió por una app de esas plataformas.
8. Columna vacía: ehail_fee está 100 % nula en verdes, así que no aporta información.
9. Duplicados: 32,790 viajes amarillos repiten proveedor, horas, zonas, distancia y total, el 0.11 %.
10. Extremos: tarifas de hasta 7,045 dólares, propinas de 766 y peajes de 1,400.

Decisiones tomadas a partir de estos resultados:

- No se borra ni modifica ningún Parquet. Las reglas se aplican en una vista con una columna calidad que indica la primera regla que falla, así se puede medir cuánto descarta cada una.
- Los nulos de Flex Fare no se imputan: Flex Fare se trata como un método de pago propio.
- Los duplicados se conservan porque los datos no tienen un identificador de viaje y dos viajes cortos idénticos son posibles. Su peso es despreciable.
- ehail_fee y request_source se conservan en la vista, pero no se usan en los indicadores.

### 3.7 y 3.8 Documentación de las consultas

Todas las consultas anteriores se ejecutaron directamente sobre los archivos Parquet. Cada una quedó en sql/ejercicio3 con su objetivo y fuente en el encabezado.

| Consulta | Objetivo | Fuente | Resultado | Decisión |
|---|---|---|---|---|
| 01_cantidad_archivos | Contar archivos por tipo | glob sobre data/raw/*/2026 | 8 amarillos y 8 verdes, de enero a agosto | El conjunto coincide con lo descargado |
| 02_cantidad_registros | Contar registros leyendo los datos | read_parquet de cada tipo | 29,703,355 amarillos y 337,114 verdes | Coincide con el manifiesto, la descarga está íntegra |
| 03_registros_metadatos | Contar registros solo con metadatos | parquet_metadata | Mismo total sin leer los datos | El conteo puede resolverse con metadatos |
| 04_columnas_por_archivo | Columnas presentes en cada archivo | parquet_schema | 21 y 22 columnas; request_source en 3 de 8 archivos por tipo | Hace falta un esquema común que tolere columnas ausentes |
| 05 y 06 tipos | Tipos lógicos de cada columna | DESCRIBE sobre read_parquet | Fechas TIMESTAMP, códigos BIGINT, montos DOUBLE | Convertir códigos a INTEGER y renombrar fechas |
| 07 y 08 muestras | Ver registros reales | USING SAMPLE con semilla 42 | Registros coherentes; cargo CBD de 0.75 en viajes al centro | Confirmar el significado de las columnas |
| 09 y 10 perfiles | Mínimos, máximos y nulos de todo | SUMMARIZE | Negativos, distancias de 328 mil millas, 25.98 % de nulos en 5 columnas | Definir reglas de calidad |
| 11 fechas | Fechas fuera del mes del archivo | read_parquet con filename | 146 amarillos y 98 verdes, desde 2001 | Regla fecha_fuera_de_archivo |
| 12 reglas | Violaciones de reglas de negocio | read_parquet de ambos tipos | 952 mil distancias cero, 372 mil duraciones no positivas, 162 mil totales negativos | Reglas de distancia, duración, monto y velocidad |
| 13 códigos | Códigos contra el diccionario | read_parquet amarillos | payment_type 0 en 7.7 millones; RatecodeID 99 en 770 mil | Tratar Flex Fare como método propio |
| 14 nulos Flex Fare | Origen de los nulos | read_parquet amarillos | Todos los nulos son payment_type 0 | No imputar, son nulos estructurales |
| 15 request_source | Explorar la columna nueva | read_parquet con filename | Existe desde junio; 23 % de amarillos con códigos HV | Incluirla en el esquema común |
| 16 duplicados | Viajes repetidos | read_parquet amarillos | 32,790 repetidos, 0.11 % | Se conservan |

### 3.9 Qué significa consultar directamente un Parquet

Consultar directamente significa que DuckDB lee el archivo en el momento de la consulta, sin copiarlo antes a una tabla ni a la memoria de Python. Parquet es columnar y guarda metadatos por row group: esquema, número de filas, mínimos y máximos por columna. DuckDB aprovecha eso de tres formas: lee solo las columnas que la consulta usa, salta row groups cuyos mínimos y máximos no cumplen el filtro y responde algunas preguntas, como el conteo de filas, solo con los metadatos. Además procesa los archivos en paralelo y por bloques, así que el volumen de datos puede superar la memoria disponible.

Con datos grandes esto evita el paso de carga, no duplica almacenamiento, permite empezar a explorar en segundos y hace que un archivo nuevo quede disponible en cuanto aparece en la carpeta. En este ejercicio los 16 archivos ocupan unos 496 MiB comprimidos, y aun así contar 30,040,469 registros o perfilar todas las columnas tomó pocos segundos.